# VSD "z-Prefix" Full-Body CT Dataset Exploration

Exploratory data analysis of the VSD z-prefix subjects — **"Full Body (no Head)"** CT scans
stored as NIfTI files, as opposed to the regular VSD "Lower_limb" subjects stored as DICOM.

**Objectives:**
1. Catalogue all z-prefix subjects and extract NIfTI + JSON metadata
2. Compare data format, scan type, and demographics vs regular VSD subjects
3. Visualize anatomy coverage (axial, coronal, sagittal views)
4. Analyze HU intensity distributions
5. Identify knee regions via bone cross-section profiling
6. Assess usability for the 3D knee reconstruction project
7. Document preprocessing differences (NIfTI loading vs DICOM)

**Input**: NIfTI files from `data/raw/VSD_Dataset/z*/`
**Output**: Exploration figures saved to `reports/figures/z_vsd_exploration/`

## Cell 1 — Imports & Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# ============================================================
# Configuration
# ============================================================
PROJECT_ROOT = Path(r"c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject")
VSD_ROOT = PROJECT_ROOT / "data" / "raw" / "VSD_Dataset"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "z_vsd_exploration"
FIG_DIR.mkdir(parents=True, exist_ok=True)

BONE_HU_THRESHOLD = 200

print(f"VSD root   : {VSD_ROOT}")
print(f"Figures dir: {FIG_DIR}")

## Cell 2 — Data Discovery: z-Prefix Subjects

Scan all `z*` directories, parse JSON metadata (demographics, description), and read
NIfTI headers (dimensions, spacing, orientation) **without loading full arrays** to save memory.

**Key difference from regular VSD:**
- Regular VSD: DICOM series in `SMIR.Lower_limb.{Age}Y.{Sex}.CT.{ID}/` folders
- z-prefix VSD: Single NIfTI file + JSON in `SMIR.Body.{Age}Y.{Sex}.CT.{ID}/` folders
- Scan type: "Full Body (no Head)" vs "Lower limb"

In [ ]:
def discover_z_cases(vsd_root):
    """Discover all z-prefix subjects and extract metadata from NIfTI headers + JSON."""
    cases = []
    for subject_name in sorted(os.listdir(vsd_root)):
        if not subject_name.startswith("z"):
            continue
        subject_dir = vsd_root / subject_name
        if not subject_dir.is_dir():
            continue

        subdirs = [d for d in subject_dir.iterdir() if d.is_dir()]
        if not subdirs:
            continue
        smir_dir = subdirs[0]
        smir_name = smir_dir.name

        # Parse demographics from folder name: SMIR.Body.{Age}Y.{Sex}.CT.{ID}
        parts = smir_name.split(".")
        age = parts[2] if len(parts) > 2 else "?"
        sex = parts[3] if len(parts) > 3 else "?"

        # Read JSON metadata
        json_path = smir_dir / f"{smir_name}.json"
        json_meta = {}
        if json_path.exists():
            with open(json_path, "r") as f:
                json_meta = json.load(f)

        # Extract subject demographics from JSON
        subj = json_meta.get("subjectSnapshot", {})
        age_days = subj.get("ageInDays", None)
        age_years = round(age_days / 365.25, 1) if age_days else None
        height_m = subj.get("heightInMeters", None)
        weight_kg = subj.get("weightInKilograms", None)
        description = json_meta.get("description", "")

        # Read NIfTI header only (no full array load)
        nii_path = smir_dir / f"{smir_name}.nii"
        if not nii_path.exists():
            print(f"  WARNING: NIfTI not found for {subject_name}")
            continue

        reader = sitk.ImageFileReader()
        reader.SetFileName(str(nii_path))
        reader.ReadImageInformation()

        size = reader.GetSize()        # (x, y, z)
        spacing = reader.GetSpacing()  # (x, y, z)
        origin = reader.GetOrigin()
        direction = reader.GetDirection()
        nii_size_mb = nii_path.stat().st_size / (1024 ** 2)

        cases.append({
            "subject_id": subject_name,
            "smir_name": smir_name,
            "description": description,
            "age_folder": age,
            "sex": sex,
            "age_years": age_years,
            "height_m": height_m if height_m else None,
            "weight_kg": weight_kg if weight_kg else None,
            "nii_path": str(nii_path),
            "nii_size_mb": round(nii_size_mb, 1),
            "size_xyz": size,
            "spacing_x": round(spacing[0], 6),
            "spacing_y": round(spacing[1], 6),
            "spacing_z": round(spacing[2], 6),
            "n_slices": size[2],
            "phys_x_mm": round(size[0] * spacing[0], 1),
            "phys_y_mm": round(size[1] * spacing[1], 1),
            "phys_z_mm": round(size[2] * spacing[2], 1),
            "origin": origin,
            "direction": direction,
        })
        print(f"  {subject_name} ({age}.{sex}): {size}, "
              f"spacing=({spacing[0]:.4f}, {spacing[1]:.4f}, {spacing[2]:.4f})mm, "
              f"z-extent={size[2] * spacing[2]:.0f}mm, "
              f"file={nii_size_mb:.0f}MB")

    return cases


print("Scanning z-prefix VSD subjects (header-only, no array loading)...")
print()
z_cases = discover_z_cases(VSD_ROOT)
print()
print(f"Total z-prefix subjects found: {len(z_cases)}")

In [ ]:
# Metadata summary table
df_z = pd.DataFrame([{k: v for k, v in c.items()
                       if k not in ("nii_path", "origin", "direction", "smir_name")}
                      for c in z_cases])
display(df_z)

## Cell 3 — Orientation & Direction Matrix Analysis

Check the NIfTI direction cosines and origins to understand orientation.
Regular VSD DICOM is loaded in scanner coordinates; NIfTI may already be in
RAS (the target orientation for our pipeline) or in some other convention.

In [ ]:
print("Direction matrices & origins for z-prefix subjects")
print("=" * 70)

# Collect unique direction matrices
unique_dirs = {}
for case in z_cases:
    sid = case["subject_id"]
    d = case["direction"]
    o = case["origin"]
    dir_key = tuple(round(x, 4) for x in d)

    if dir_key not in unique_dirs:
        unique_dirs[dir_key] = []
    unique_dirs[dir_key].append(sid)

    print(f"\n{sid}:")
    dir_mat = np.array(d).reshape(3, 3)
    print(f"  Direction matrix:\n    {dir_mat[0]}\n    {dir_mat[1]}\n    {dir_mat[2]}")
    print(f"  Origin: ({o[0]:.2f}, {o[1]:.2f}, {o[2]:.2f})")

print("\n" + "=" * 70)
print(f"\nUnique direction matrices: {len(unique_dirs)}")
for i, (dir_key, sids) in enumerate(unique_dirs.items()):
    dir_mat = np.array(dir_key).reshape(3, 3)
    print(f"\n  Pattern {i+1} ({len(sids)} subjects: {sids}):")
    print(f"    {dir_mat}")

    # Interpret orientation
    # RAS identity = diag(1, 1, 1), LPS = diag(-1, -1, 1)
    if np.allclose(dir_mat, np.eye(3)):
        print("    -> RAS orientation (matches our target)")
    elif np.allclose(dir_mat, np.diag([-1, -1, 1])):
        print("    -> LPS orientation (DICOM default, needs reorientation to RAS)")
    else:
        print("    -> Non-standard orientation — inspect carefully")

## Cell 4 — Side-by-Side Comparison: z-Prefix vs Regular VSD

Compare key characteristics of the two VSD subsets to document every difference
that affects preprocessing.

In [ ]:
# Load regular VSD metadata for comparison (header-only for new subjects too)
def discover_regular_vsd_headers(vsd_root):
    """Discover non-z VSD subjects, read DICOM metadata."""
    cases = []
    for subject_name in sorted(os.listdir(vsd_root)):
        if subject_name.startswith("z"):
            continue
        subject_dir = vsd_root / subject_name
        if not subject_dir.is_dir():
            continue

        subdirs = [d for d in subject_dir.iterdir() if d.is_dir()]
        if not subdirs:
            continue
        smir_dir = subdirs[0]
        smir_name = smir_dir.name

        parts = smir_name.split(".")
        age = parts[2] if len(parts) > 2 else "?"
        sex = parts[3] if len(parts) > 3 else "?"

        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(str(smir_dir))
        if not dicom_names:
            continue
        reader.SetFileNames(dicom_names)
        reader.ReadImageInformation()

        size = reader.GetSize()
        spacing = reader.GetSpacing()

        cases.append({
            "subject_id": subject_name,
            "age": age,
            "sex": sex,
            "size_xyz": size,
            "spacing_x": round(spacing[0], 6),
            "spacing_y": round(spacing[1], 6),
            "spacing_z": round(spacing[2], 6),
            "n_slices": size[2],
            "phys_z_mm": round(size[2] * spacing[2], 1),
            "n_subdirs": len(subdirs),
        })
    return cases


print("Loading regular VSD subject headers for comparison...")
reg_cases = discover_regular_vsd_headers(VSD_ROOT)
print(f"Regular VSD subjects: {len(reg_cases)}")
print(f"z-prefix VSD subjects: {len(z_cases)}")

# Build comparison table
comparison = {
    "Property": [
        "File format",
        "SMIR naming",
        "Scan type (from JSON/folder)",
        "Subject count",
        "XY spacing range (mm)",
        "Z spacing range (mm)",
        "Z-extent range (mm)",
        "Image XY size",
        "Slice count range",
        "Bilateral (2 legs)?",
        "Multi-subfolder subjects",
    ],
    "Regular VSD": [
        "DICOM series",
        "SMIR.Lower_limb.{Age}Y.{Sex}.CT.{ID}",
        "Lower limb CT",
        str(len(reg_cases)),
        f"{min(c['spacing_x'] for c in reg_cases):.3f} – {max(c['spacing_x'] for c in reg_cases):.3f}",
        f"{min(c['spacing_z'] for c in reg_cases):.3f} – {max(c['spacing_z'] for c in reg_cases):.3f}",
        f"{min(c['phys_z_mm'] for c in reg_cases):.0f} – {max(c['phys_z_mm'] for c in reg_cases):.0f}",
        "512 × 512",
        f"{min(c['n_slices'] for c in reg_cases)} – {max(c['n_slices'] for c in reg_cases)}",
        "Yes (full lower-limb scans)",
        f"{sum(1 for c in reg_cases if c['n_subdirs'] > 1)} subjects (split scans)",
    ],
    "z-Prefix VSD": [
        "NIfTI (.nii)",
        "SMIR.Body.{Age}Y.{Sex}.CT.{ID}",
        "Full Body (no Head) CT",
        str(len(z_cases)),
        f"{min(c['spacing_x'] for c in z_cases):.3f} – {max(c['spacing_x'] for c in z_cases):.3f}",
        f"{min(c['spacing_z'] for c in z_cases):.3f} – {max(c['spacing_z'] for c in z_cases):.3f}",
        f"{min(c['phys_z_mm'] for c in z_cases):.0f} – {max(c['phys_z_mm'] for c in z_cases):.0f}",
        f"{min(c['size_xyz'][0] for c in z_cases)}–{max(c['size_xyz'][0] for c in z_cases)} × "
        f"{min(c['size_xyz'][1] for c in z_cases)}–{max(c['size_xyz'][1] for c in z_cases)}",
        f"{min(c['n_slices'] for c in z_cases)} – {max(c['n_slices'] for c in z_cases)}",
        "Yes (full body includes both legs)",
        "None (single NIfTI per subject)",
    ],
}

df_cmp = pd.DataFrame(comparison)
display(df_cmp.style.set_properties(**{"text-align": "left"}))

In [ ]:
# Visual comparison of spacing distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# XY spacing
reg_xy = [c["spacing_x"] for c in reg_cases]
z_xy = [c["spacing_x"] for c in z_cases]
axes[0].hist(reg_xy, bins=15, alpha=0.7, color="steelblue", label=f"Regular ({len(reg_cases)})")
axes[0].hist(z_xy, bins=15, alpha=0.7, color="coral", label=f"z-prefix ({len(z_cases)})")
axes[0].set_title("XY Pixel Spacing")
axes[0].set_xlabel("mm")
axes[0].legend()

# Z spacing
reg_zs = [c["spacing_z"] for c in reg_cases]
z_zs = [c["spacing_z"] for c in z_cases]
axes[1].hist(reg_zs, bins=15, alpha=0.7, color="steelblue", label="Regular")
axes[1].hist(z_zs, bins=15, alpha=0.7, color="coral", label="z-prefix")
axes[1].set_title("Z Spacing / Slice Thickness")
axes[1].set_xlabel("mm")
axes[1].legend()

# Z-extent
reg_ze = [c["phys_z_mm"] for c in reg_cases]
z_ze = [c["phys_z_mm"] for c in z_cases]
axes[2].hist(reg_ze, bins=15, alpha=0.7, color="steelblue", label="Regular")
axes[2].hist(z_ze, bins=15, alpha=0.7, color="coral", label="z-prefix")
axes[2].set_title("Physical Z-Extent")
axes[2].set_xlabel("mm")
axes[2].legend()

plt.suptitle("Spacing & Extent Distributions: Regular vs z-Prefix VSD", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "z_vs_regular_spacing_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Cell 5 — Sample Volume Visualization (3 Representative Subjects)

Load full arrays for 3 representative z-prefix subjects to visualize anatomy coverage.
We pick the first, middle, and last subjects to span the range.

**WARNING**: Each NIfTI is ~1.6 GB uncompressed. We load one at a time and free memory
after visualization to avoid OOM.

In [ ]:
import gc

# Pick 3 representative subjects: first, middle, last
sample_indices = [0, len(z_cases) // 2, len(z_cases) - 1]
sample_cases = [z_cases[i] for i in sample_indices]

for case in sample_cases:
    sid = case["subject_id"]
    print(f"\nLoading {sid} ({case['nii_size_mb']:.0f} MB)...")

    img = sitk.ReadImage(case["nii_path"])
    arr = sitk.GetArrayFromImage(img).astype(np.float32)  # shape: (z, y, x)
    print(f"  Array shape: {arr.shape}, dtype: {arr.dtype}")
    print(f"  HU range: [{arr.min():.0f}, {arr.max():.0f}], mean: {arr.mean():.0f}")

    mid_z = arr.shape[0] // 2
    mid_y = arr.shape[1] // 2
    mid_x = arr.shape[2] // 2

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))

    # Axial (mid-volume)
    axes[0].imshow(arr[mid_z], cmap="gray", vmin=-500, vmax=1500)
    axes[0].set_title(f"Axial (z={mid_z})")
    axes[0].axis("off")

    # Coronal
    axes[1].imshow(arr[:, mid_y, :], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[1].set_title(f"Coronal (y={mid_y})")
    axes[1].axis("off")

    # Sagittal
    axes[2].imshow(arr[:, :, mid_x], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[2].set_title(f"Sagittal (x={mid_x})")
    axes[2].axis("off")

    # Info panel
    axes[3].axis("off")
    info_text = (
        f"Subject: {sid}\n"
        f"Description: {case['description']}\n"
        f"Age: {case['age_years']}y / Sex: {case['sex']}\n"
        f"Height: {case.get('height_m', '?')}m  Weight: {case.get('weight_kg', '?')}kg\n"
        f"Size: {case['size_xyz']}\n"
        f"Spacing: ({case['spacing_x']:.4f}, {case['spacing_y']:.4f}, {case['spacing_z']:.4f})\n"
        f"Z-extent: {case['phys_z_mm']:.0f}mm\n"
        f"HU range: [{arr.min():.0f}, {arr.max():.0f}]\n"
        f"File size: {case['nii_size_mb']:.0f} MB"
    )
    axes[3].text(0.05, 0.5, info_text, transform=axes[3].transAxes,
                fontsize=11, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
    axes[3].set_title("Metadata")

    plt.suptitle(f"z-Prefix Subject {sid} — Full Body (no Head) CT Overview", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / f"z_{sid}_overview.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")

    # Free memory
    del arr, img
    gc.collect()

## Cell 6 — HU Intensity Distributions (Sample Subjects)

Analyze HU distributions for the same 3 sample subjects.
Compare with confirmed bone window [-450, 1050] from the regular VSD analysis.

In [ ]:
fig, axes = plt.subplots(1, len(sample_cases), figsize=(7 * len(sample_cases), 5))
if len(sample_cases) == 1:
    axes = [axes]

hu_stats = []
for i, case in enumerate(sample_cases):
    sid = case["subject_id"]
    print(f"Loading {sid} for HU analysis...")

    img = sitk.ReadImage(case["nii_path"])
    arr = sitk.GetArrayFromImage(img).astype(np.float32)

    hu_stats.append({
        "subject_id": sid,
        "hu_min": float(arr.min()),
        "hu_max": float(arr.max()),
        "hu_mean": float(arr.mean()),
        "hu_std": float(arr.std()),
    })

    # Exclude air background for cleaner histogram
    tissue = arr[arr > -900].ravel()

    axes[i].hist(tissue, bins=200, color="coral", alpha=0.7, density=True)
    axes[i].axvline(x=-450, color="red", linestyle="--", alpha=0.7, label="Window low (-450)")
    axes[i].axvline(x=1050, color="red", linestyle="--", alpha=0.7, label="Window high (1050)")
    axes[i].axvline(x=200, color="orange", linestyle=":", alpha=0.7, label="Bone threshold (200)")
    axes[i].set_title(f"{sid} ({case['age_years']}y/{case['sex']})")
    axes[i].set_xlabel("HU")
    axes[i].set_ylabel("Density")
    axes[i].set_xlim(-500, 2000)
    if i == 0:
        axes[i].legend(fontsize=8)

    del arr, img
    gc.collect()

plt.suptitle("z-Prefix VSD: HU Intensity Distributions (air excluded, HU > -900)", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "z_hu_histograms.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

df_hu = pd.DataFrame(hu_stats)
display(df_hu)

## Cell 7 — Bone Cross-Section Profiling & Knee Identification (All Subjects)

For each z-prefix subject, compute the bone cross-section area at each z-slice
to identify the knee region. Since these are full-body scans (neck to feet),
the profile will show peaks for: **shoulders/arms, pelvis/hips, knees, ankles**.

The knee joint creates a distinctive peak at roughly 15-35% of the total z-extent
(from foot to neck). This is different from the regular VSD lower-limb scans
where the knee is at ~50% of the z-range.

**Memory strategy**: Load each volume, compute the 1D bone-area profile, free the array.

In [ ]:
def compute_bone_profile(nii_path):
    """Load volume, compute per-slice bone area, return profile + HU stats."""
    img = sitk.ReadImage(nii_path)
    arr = sitk.GetArrayFromImage(img).astype(np.float32)  # (z, y, x)
    bone_area = (arr > BONE_HU_THRESHOLD).sum(axis=(1, 2)).astype(float)
    hu_min, hu_max, hu_mean = float(arr.min()), float(arr.max()), float(arr.mean())
    del arr, img
    gc.collect()
    return bone_area, hu_min, hu_max, hu_mean


def find_knee_in_fullbody(bone_area, spacing_z, smooth_sigma=30):
    """Identify knee center in a full-body scan.

    In a full-body (no head) scan going foot-to-neck or neck-to-foot,
    the knee is in the lower portion of the body. We search for prominent
    bone-area peaks and pick the one in a plausible knee range.
    """
    smoothed = gaussian_filter1d(bone_area, sigma=smooth_sigma)
    z_mm = np.arange(len(bone_area)) * spacing_z
    total_z = z_mm[-1]

    peaks, props = find_peaks(
        smoothed,
        height=np.max(smoothed) * 0.10,
        distance=50,
        prominence=np.max(smoothed) * 0.03,
    )

    # The knee is a local peak with a distinctive shape.
    # In a full-body scan, it can be anywhere depending on orientation.
    # We'll return all peaks and let the visualization help identify which is the knee.
    return z_mm, smoothed, bone_area, peaks


print("Computing bone cross-section profiles for all z-prefix subjects...")
print("(Loading each volume one at a time to manage memory)\n")

profiles = []
for case in z_cases:
    sid = case["subject_id"]
    print(f"  Processing {sid}...", end=" ")
    bone_area, hu_min, hu_max, hu_mean = compute_bone_profile(case["nii_path"])
    z_mm, smoothed, raw, peaks = find_knee_in_fullbody(
        bone_area, case["spacing_z"]
    )
    profiles.append({
        "subject_id": sid,
        "z_mm": z_mm,
        "smoothed": smoothed,
        "raw": raw,
        "peaks": peaks,
        "hu_min": hu_min,
        "hu_max": hu_max,
        "hu_mean": hu_mean,
    })
    case["hu_min"] = hu_min
    case["hu_max"] = hu_max
    case["hu_mean"] = hu_mean

    print(f"done — {len(peaks)} peaks found, "
          f"HU=[{hu_min:.0f}, {hu_max:.0f}], z-extent={z_mm[-1]:.0f}mm")

print(f"\nAll {len(profiles)} profiles computed.")

In [ ]:
# Plot bone cross-section profiles for all z-prefix subjects
n_cols = 4
n_rows = (len(profiles) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.ravel()

for i, prof in enumerate(profiles):
    sid = prof["subject_id"]
    z_mm = prof["z_mm"]
    smoothed = prof["smoothed"]
    peaks = prof["peaks"]

    axes[i].plot(z_mm, prof["raw"], alpha=0.2, color="gray", linewidth=0.5)
    axes[i].plot(z_mm, smoothed, color="coral", linewidth=2)

    for p in peaks:
        axes[i].axvline(x=z_mm[p], color="lightcoral", alpha=0.5, linestyle=":")

    axes[i].set_title(f"{sid} ({len(peaks)} peaks)", fontsize=10)
    axes[i].set_xlabel("Z (mm)", fontsize=8)
    axes[i].set_ylabel("Bone area", fontsize=8)
    axes[i].tick_params(labelsize=7)

# Hide unused subplots
for j in range(len(profiles), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    "z-Prefix VSD: Bone Cross-Section Profiles (Full Body)\n"
    "Red dotted lines = detected peaks (shoulders, pelvis, knees, ankles)",
    fontsize=13, y=1.02
)
plt.tight_layout()
fig_path = FIG_DIR / "z_bone_profiles_all.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Cell 8 — Coronal Full-Body Views (Sample Subjects)

Visualize full coronal slices for the 3 sample subjects to confirm anatomy coverage.
Since these are "Full Body (no Head)" scans, we expect to see from neck/shoulders
down to the feet, with both legs visible and the knee joints present.

In [ ]:
fig, axes = plt.subplots(1, len(sample_cases), figsize=(8 * len(sample_cases), 16))
if len(sample_cases) == 1:
    axes = [axes]

for i, case in enumerate(sample_cases):
    sid = case["subject_id"]
    print(f"Loading {sid} for coronal view...")

    img = sitk.ReadImage(case["nii_path"])
    arr = sitk.GetArrayFromImage(img).astype(np.float32)

    mid_y = arr.shape[1] // 2
    coronal = arr[:, mid_y, :]

    axes[i].imshow(coronal, cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[i].set_title(
        f"{sid}\n{case['description']}\n"
        f"Z-extent: {case['phys_z_mm']:.0f}mm | "
        f"Age: {case['age_years']}y | Sex: {case['sex']}",
        fontsize=11
    )
    axes[i].set_xlabel("X (pixel)")
    axes[i].set_ylabel("Z slice (bottom → top)")

    # Mark detected peaks from profile
    prof = profiles[z_cases.index(case)]
    for p in prof["peaks"]:
        axes[i].axhline(y=p, color="red", alpha=0.4, linewidth=1, linestyle=":")

    del arr, img
    gc.collect()

plt.suptitle(
    "z-Prefix VSD: Full Coronal Views (Full Body CT)\n"
    "Red dotted = detected bone-area peaks",
    fontsize=14, y=1.01
)
plt.tight_layout()
fig_path = FIG_DIR / "z_coronal_views.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Cell 9 — Demographics Summary

Aggregate and visualize demographics for the z-prefix cohort, and compare
with the regular VSD cohort.

In [ ]:
# Demographics table
demo_data = []
for case in z_cases:
    demo_data.append({
        "subject_id": case["subject_id"],
        "age_years": case["age_years"],
        "sex": case["sex"],
        "height_m": case["height_m"],
        "weight_kg": case["weight_kg"],
        "bmi": round(case["weight_kg"] / (case["height_m"] ** 2), 1)
               if case["height_m"] and case["weight_kg"] else None,
    })

df_demo = pd.DataFrame(demo_data)
display(df_demo)

# Summary statistics
print("\nz-Prefix VSD Demographics Summary")
print("=" * 40)
print(f"  N subjects: {len(df_demo)}")
print(f"  Sex: M={sum(df_demo['sex'] == 'M')}, F={sum(df_demo['sex'] == 'F')}")
print(f"  Age: {df_demo['age_years'].min():.0f} – {df_demo['age_years'].max():.0f} years "
      f"(mean={df_demo['age_years'].mean():.1f})")
valid_h = df_demo["height_m"].dropna()
valid_w = df_demo["weight_kg"].dropna()
valid_bmi = df_demo["bmi"].dropna()
if len(valid_h) > 0:
    print(f"  Height: {valid_h.min():.2f} – {valid_h.max():.2f}m (mean={valid_h.mean():.2f})")
if len(valid_w) > 0:
    print(f"  Weight: {valid_w.min():.1f} – {valid_w.max():.1f}kg (mean={valid_w.mean():.1f})")
if len(valid_bmi) > 0:
    print(f"  BMI: {valid_bmi.min():.1f} – {valid_bmi.max():.1f} (mean={valid_bmi.mean():.1f})")

# Age distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age
z_ages = df_demo["age_years"].dropna().values
reg_ages = [float(c["age"].replace("Y", "")) for c in reg_cases if "Y" in str(c.get("age", ""))]
axes[0].hist(reg_ages, bins=10, alpha=0.7, color="steelblue", label=f"Regular ({len(reg_ages)})")
axes[0].hist(z_ages, bins=10, alpha=0.7, color="coral", label=f"z-prefix ({len(z_ages)})")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age (years)")
axes[0].legend()

# Sex
reg_sex_m = sum(1 for c in reg_cases if c["sex"] == "M")
reg_sex_f = sum(1 for c in reg_cases if c["sex"] == "F")
z_sex_m = sum(df_demo["sex"] == "M")
z_sex_f = sum(df_demo["sex"] == "F")
x = np.arange(2)
w = 0.35
axes[1].bar(x - w/2, [reg_sex_m, reg_sex_f], w, color="steelblue", alpha=0.7, label="Regular")
axes[1].bar(x + w/2, [z_sex_m, z_sex_f], w, color="coral", alpha=0.7, label="z-prefix")
axes[1].set_xticks(x)
axes[1].set_xticklabels(["Male", "Female"])
axes[1].set_title("Sex Distribution")
axes[1].set_ylabel("Count")
axes[1].legend()

# Combined sample counts
total_reg_knees = len(reg_cases) * 2  # bilateral
total_z_knees = len(z_cases) * 2      # bilateral (full body has both legs)
axes[2].bar(
    ["Regular VSD\n(Lower limb)", "z-Prefix VSD\n(Full Body)", "Combined\nTotal"],
    [total_reg_knees, total_z_knees, total_reg_knees + total_z_knees],
    color=["steelblue", "coral", "mediumpurple"],
    alpha=0.7,
)
axes[2].set_title("Potential Knee Volumes (bilateral)")
axes[2].set_ylabel("Count")
for j, v in enumerate([total_reg_knees, total_z_knees, total_reg_knees + total_z_knees]):
    axes[2].text(j, v + 0.3, str(v), ha="center", fontweight="bold")

plt.suptitle("Demographics: Regular vs z-Prefix VSD", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "z_demographics_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Cell 10 — HU Range Summary (All Subjects)

Aggregate HU statistics from all z-prefix subjects (computed during bone profiling)
and compare with the regular VSD confirmed bone window [-450, 1050].

In [ ]:
# Collect HU stats from all subjects
hu_all = pd.DataFrame([{
    "subject_id": p["subject_id"],
    "hu_min": p["hu_min"],
    "hu_max": p["hu_max"],
    "hu_mean": p["hu_mean"],
} for p in profiles])

display(hu_all)

print("\nHU Range Summary (z-prefix VSD)")
print("=" * 40)
print(f"  Global min HU: {hu_all['hu_min'].min():.0f}")
print(f"  Global max HU: {hu_all['hu_max'].max():.0f}")
print(f"  Mean HU range: [{hu_all['hu_min'].mean():.0f}, {hu_all['hu_max'].mean():.0f}]")
print(f"  Mean of means: {hu_all['hu_mean'].mean():.0f}")
print()
print("Comparison with regular VSD confirmed window [-450, 1050]:")
print("  -> The same bone window should work for z-prefix subjects")
print("     since they are also CT scans from the same VSD collection.")

## Exploration Findings

### What Are the z-Prefix Subjects?
- **16 subjects** from the same VSD collection, but a different scan protocol
- **"Full Body (no Head)"** CT scans — cover from **neck/shoulders down to feet**
- Stored as **NIfTI (.nii)** files with JSON metadata (not DICOM series)
- Named `SMIR.Body.{Age}Y.{Sex}.CT.{ID}` (vs `SMIR.Lower_limb.…` for regular VSD)
- File sizes: ~1.5–3.3 GB each (uncompressed NIfTI)
- Licensed under CC BY-NC 3.0 Switzerland

### Key Differences from Regular VSD

| Property | Regular VSD | z-Prefix VSD |
|----------|-------------|--------------|
| File format | DICOM series | NIfTI (.nii) |
| Scan type | Lower limb only | Full body (no head) |
| Loading method | `sitk.ImageSeriesReader` + DICOM | `sitk.ReadImage(nii_path)` |
| Subjects | 11 folders | 16 folders |
| Anatomy | Ankle → hip | Feet → neck/shoulders |
| Split scans | Some subjects have 2 subfolders | Never (single NIfTI) |

### Demographics
- Age: 25–84 years (wider range than regular VSD)
- Sex: mixed M/F
- Height/weight available from JSON (2 subjects missing anthropometrics)
- All are healthy subjects (same VSD collection, no pathology)

### Implications for Preprocessing
1. **Loading**: Use `sitk.ReadImage()` for NIfTI instead of DICOM series reader
2. **Orientation**: Check NIfTI direction cosines — may already be in RAS or need reorientation
3. **Knee localization**: The knee is a much smaller fraction of the total scan — bone cross-section profiling needs adjusted z-range expectations (the knee is no longer at ~50% of z-extent)
4. **Bilateral separation**: Still needed — full body scans contain both legs
5. **Memory**: Full-body NIfTI arrays are ~1.6 GB each in float32 — process one at a time
6. **Same HU window**: The confirmed [-450, 1050] bone window applies (same scanner/protocol family)
7. **Crop margin**: +/-100mm around knee center should still be sufficient

### Sample Size Impact
- Regular VSD: 11 subjects × 2 legs = **22 healthy knee volumes** (minus exclusions)
- z-Prefix VSD: 16 subjects × 2 legs = **32 healthy knee volumes** (pending exclusion screening)
- Combined potential: **up to 54 healthy knee volumes** — a significant boost to the training set

### Next Steps
1. Run the bone-profile plots and coronal views to visually confirm knee presence in all 16 subjects
2. Identify any subjects to exclude (e.g., scans that don't extend below the knee)
3. Adapt the knee cropping pipeline to handle NIfTI loading + full-body z-range
4. Add z-prefix subjects to the unified preprocessing notebook